<a href="https://colab.research.google.com/github/Bilra-Hkr/BigData26_B_2411533007_DerielChaerahman/blob/main/Praktikum5/BD_P05_Latihan_2411533007_DerielChaerahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# H-1. Peta Pipeline dan Permainan Kartu Kantin

In [1]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

# H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal

In [2]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom  :", df_sumber.shape)
print("Kolom         :", list(df_sumber.columns))

import http.server, urllib.parse, html

HALAMAN_UKURAN = 100          # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):          # matikan log
            pass
        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Ukuran berkas : 16.1 MB
Baris, kolom  : (91017, 4)
Kolom         : ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


# H-3. Crawling Portal Lokal: Latihan Aman


membaca robots.txt. Memeriksa halaman
berita biasa dan /arsip-privat/ yang dilarang oleh portal.

In [3]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


Melakukan parse_halaman mengubah HTML menjadi data.

In [4]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
             "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


# H-4. Crawling Berita Sungguhan dari Sitemap detik.com

In [5]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import re

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("news:title"), u.find("news:publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else None,  # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True) if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

In [6]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa
diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>

<news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan
DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
 <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari
lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak
Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat
banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan
Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) ==
parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29T12:52:00+00:00'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29T12:50:00+00:00'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29T12:44:00+00:00'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


In [7]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: deril@gmail.com)"
DELAY = 2.0                 # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7        # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20            # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://finance.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content            # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

In [39]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul   :", len(cek))
    print("Judul kosong      :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat      :", cek["url"].duplicated().sum())
    print("Tanggal kosong    :", cek["tanggal"].isna().sum())
    print("Kategori          :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul   : 20
Judul kosong      : 0
URL duplikat      : 0
Tanggal kosong    : 0
Kategori          : {'news': 7, 'inet': 7, 'sport': 6}
                                                                            judul kategori                   tanggal
Kemenkop Dorong Perajin NTB Berkoperasi, Perkuat Ekosistem Wastra &amp;amp; Kriya     news 2026-11-12T16:53:00+07:00
                      Kata-kata Gibran Saat Peluk Prabowo: Saya Minta Maaf Ya Pak     news 2026-11-12T16:48:00+07:00
          KEK Sanur Beri Dampak Ekonomi Rp 1,13 Triliun, Dikunjungi 36.798 Pasien     news 2026-11-12T16:46:00+07:00


In [41]:
cek6 = pd.DataFrame(sampel)
print("Kategori yang muncul:", cek6["kategori"].value_counts().to_dict())
for nama, lulus in aturan_kualitas(cek6).items():
       print(f"{nama:<24} gagal: {(~lulus).sum():>2} dari {len(lulus)}")

Kategori yang muncul: {'news': 7, 'inet': 7, 'sport': 6}
judul tidak kosong       gagal:  0 dari 20
judul 3-30 kata          gagal:  0 dari 20
kategori dikenal         gagal:  0 dari 20
tanggal valid            gagal: 20 dari 20
url artikel valid        gagal:  0 dari 20
judul tidak duplikat     gagal:  0 dari 20


# H-5. Channel ala Flume ke lapisan_mentah

In [9]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)                       # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0
print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3],
"...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 57.2 detik untuk 911 halaman


In [38]:
def ukur_crawl(n_hal, delay):
    t0 = time.perf_counter()
    n = sum(1 for _ in crawl_iter(BASE, range(1, n_hal + 1), delay=delay))
    return n, time.perf_counter() - t0

# Pembanding: 50 halaman tanpa jeda, lalu dengan jeda 0,05 detik
n0, w0 = ukur_crawl(50, 0.0)
n1, w1 = ukur_crawl(50, 0.05)
print(f"50 halaman, tanpa jeda : {w0:.2f} dtk ({n0:,} judul)")
print(f"50 halaman, jeda 0,05  : {w1:.2f} dtk ({n1:,} judul)")

# Estimasi 911 halaman dengan jeda 1 detik (dihitung, tidak dijalankan)
kerja = w0 / 50                       # detik kerja per halaman tanpa jeda
est = N_HALAMAN * (kerja + 1.0)
print(f"Estimasi 911 halaman, jeda 1 dtk: {est:.0f} dtk = {est/60:.1f} menit")

50 halaman, tanpa jeda : 2.61 dtk (5,000 judul)
50 halaman, jeda 0,05  : 6.69 dtk (5,000 judul)
Estimasi 911 halaman, jeda 1 dtk: 959 dtk = 16.0 menit


# H-6. Quality, Cleaning, dan Parquet Berpartisi (15 menit)

In [10]:
import pandas as pd
import re

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")
KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"    : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"     : jml_kata.between(3, 30),
        "kategori dikenal"      : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"         : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"     : d["url"].str.match(POLA_ARTIKEL),
        "judul tidak duplikat"  : ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,}  ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0  ( 0.00%)
judul 3-30 kata          gagal:     78  ( 0.09%)
kategori dikenal         gagal:      0  ( 0.00%)
tanggal valid            gagal:      0  ( 0.00%)
url artikel valid        gagal:  5,640  ( 6.20%)
judul tidak duplikat     gagal:    235  ( 0.26%)


In [11]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")   # kosong bila url rusak
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


In [42]:
def mapper_kunci(record):
    """Satu berita -> pasangan (kata kunci, 1)."""
    for kw in record.get("kata_kunci", "").split(","):
        kw = kw.strip().lower()
        if kw:                      # lewati kata kunci kosong
            yield (kw, 1)

def reducer_kunci(pasangan):
    """Jumlahkan per kata kunci."""
    hasil = collections.Counter()
    for kw, n in pasangan:
        hasil[kw] += n
    return hasil

pasangan = [p for rec in sampel for p in mapper_kunci(rec)]
print("Berita diproses         :", len(sampel))
print("Pasangan (kata kunci, 1):", len(pasangan))
print("Kata kunci terbanyak    :")
for kw, n in reducer_kunci(pasangan).most_common(10):
    print(f"  {kw:<25} {n}")

Berita diproses         : 20
Pasangan (kata kunci, 1): 87
Kata kunci terbanyak    :
  jensen huang              4
  gibran rakabuming         2
  ceo nvidia                2
  perselingkuhan            2
  kimberly powell           2
  nvidia                    2
  basket                    2
  ibl all indonesian 2026   2
  ibl all indonesian        2
  koperasi                  1


In [12]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan dua pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


# H-7. Query DuckDB dan Bentuk Dokumen

In [13]:
import duckdb
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


# H-8. Mini MapReduce: Kasus Kantin

In [14]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],      # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],      # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


# H-9. Word Count Terdistribusi: Partisi sebagai Input Split

In [15]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")
top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


# H-10. Combiner dan Jumlah Data yang Berpindah

In [16]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")

for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for
x in h):>20,}")
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for
x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.520              825,601
1 proses, dengan combiner         0.598               74,009
2 proses, tanpa combiner          1.163              825,601
2 proses, dengan combiner         1.174               74,009

Hasil dengan combiner sama persis: True


# H-11. Data Skew dan Beban Reducer

In [35]:
R = 4        # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", " ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (2, 8, 32):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    2 reducer: terberat 52.27% | bagian adil 50.00%
    8 reducer: terberat 16.83% | bagian adil 12.50%
   32 reducer: terberat  5.82% | bagian adil  3.12%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


In [36]:
#Latihan n-4 : Tabel untuk laporan

baris_tab = []
for r in (2, 8, 32):
    b = beban(hasil_map, r)
    baris_tab.append({
        "R": r,
        "terberat (%)": round(max(b) / sum(b) * 100, 2),
        "bagian adil (%)": round(100 / r, 2),
        "rasio": round(max(b) / sum(b) / (1 / r), 2),
    })
print(pd.DataFrame(baris_tab).to_string(index=False))

 R  terberat (%)  bagian adil (%)  rasio
 2         52.27            50.00   1.05
 8         16.83            12.50   1.35
32          5.82             3.12   1.86


# H-12. Hadoop Streaming

In [18]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
          print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')
print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


In [19]:
!head -n 3 masukan.tsv | python3 mapper.py

!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m2.291s
user	0m2.352s
sys	0m0.084s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


In [20]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


### (Opsional) Hadoop Streaming asli, local mode
Sel di bawah memakan waktu (unduh ~700 MB). Jalankan hanya jika waktu tersisa, lalu lampirkan log dan hasil `diff`.

In [21]:
%%bash
sudo apt-get install -y openjdk-11-jdk-headless
wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
tar -xzf hadoop-3.3.6.tar.gz
export HADOOP_HOME=$PWD/hadoop-3.3.6
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64

rm -rf keluaran_hadoop
$HADOOP_HOME/bin/hadoop jar \
  $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input masukan.tsv -output keluaran_hadoop

ls keluaran_hadoop
diff <(LC_ALL=C sort hasil_pipe.txt) keluaran_hadoop/part-00000 && echo "SAMA"

Process is interrupted.


# H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman

In [32]:
def mapper_bulan(path):
    """Key gabungan (bulan, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for t, b in zip(df_["judul"], df_["bulan"]):
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(b, kata)] += 1
    return c

gabung_bulan = collections.Counter()
for c in map(mapper_bulan, splits):
    gabung_bulan.update(c)

per_bulan = collections.defaultdict(list)
for (bln, kata), n in gabung_bulan.items():
    per_bulan[bln].append((n, kata))

for bln in sorted(per_bulan):
    top5 = sorted(per_bulan[bln], reverse=True)[:5]
    print(f"{bln}", ", ".join(f"{k} ({n:,})" for n, k in top5))

# Perbadingan : kata yang berubah dari Januari ke Mei (top 10)
top10 = lambda b: {k for n, k in sorted(per_bulan[b], reverse=True)[:10]}
print("\nHanya di top-10 Januari:", sorted(top10("2020-01") - top10("2020-05")))
print("Hanya di top-10 Mei    :", sorted(top10("2020-05") - top10("2020-01")))

2020-01 banjir (1,220), polisi (554), jokowi (516), tahun (493), baru (475)
2020-02 corona (1,076), polisi (630), virus (596), banjir (572), anak (453)
2020-03 corona (6,986), virus (1,340), positif (925), pasien (749), rumah (695)
2020-04 corona (4,914), positif (828), pandemi (706), psbb (689), warga (609)
2020-05 corona (2,935), kasus (640), positif (628), pandemi (596), warga (565)
2020-06 corona (943), normal (471), new (465), kasus (315), juni (262)

Hanya di top-10 Januari: ['banjir', 'baru', 'china', 'indonesia', 'jokowi', 'kpk', 'minta', 'mobil', 'tahun']
Hanya di top-10 Mei    : ['corona', 'covid', 'kasus', 'lebaran', 'normal', 'pandemi', 'positif', 'psbb', 'warga']


In [33]:
def mapper_bigram(path):
    """Mapper bigram: kembalikan (jumlah pasangan mentah, Counter hasil combiner)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    pasangan, c = 0, collections.Counter()
    for t in judul:
        tok = tokenisasi(t)
        for a, b in zip(tok, tok[1:]):      # pasangan kata berurutan
            pasangan += 1                   # sebelum combiner: 1 pasangan = 1 baris
            c[f"{a} {b}"] += 1              # combiner: jumlahkan di sisi mapper
    return pasangan, c

hasil_bg = [mapper_bigram(p) for p in splits]
tanpa = sum(x for x, _ in hasil_bg)
dengan = sum(len(c) for _, c in hasil_bg)
print("Bigram tanpa combiner :", f"{tanpa:,}")
print("Bigram dengan combiner:", f"{dengan:,}", f"(turun {tanpa/dengan:.1f}x)")

total_bg = collections.Counter()
for _, c in hasil_bg:
    total_bg.update(c)
print("5 bigram teratas:", total_bg.most_common(5))
print("'virus corona':", f"{total_bg['virus corona']:,}")

Bigram tanpa combiner : 734,783
Bigram dengan combiner: 411,217 (turun 1.8x)
5 bigram teratas: [('virus corona', 2915), ('corona di', 2178), ('positif corona', 1804), ('di tengah', 1258), ('pandemi corona', 1075)]
'virus corona': 2,915


# O. Tugas Mandiri: Inverted Index dengan MapReduce
Untuk setiap kata, simpan daftar id judul yang memuatnya.

**Penanganan id kosong (kategori `inet`):** kolom `id` kosong karena URL-nya rusak (`#`). Baris tidak dibuang agar satu kategori utuh tidak hilang. Sebagai gantinya, baris itu diberi **id pengganti** berbentuk `inet-000001`, `inet-000002`, dan seterusnya (kategori + nomor urut baris dalam partisi). Awalan kategori mencegah bentrok dengan id asli, dan id pengganti tetap bisa dilacak kembali ke barisnya.

In [23]:
def id_dokumen(nilai_id, kategori, nomor):
    """id asli bila ada; bila kosong, buat id pengganti '<kategori>-<nomor>'."""
    return int(nilai_id) if pd.notna(nilai_id) else f"{kategori}-{nomor:06d}"

def kunci_urut(x):
    """Id angka diurutkan sebagai angka dulu, lalu id pengganti (teks)."""
    return (1, 0, x) if isinstance(x, str) else (0, x, "")

def mapper_index(path):
    """Mapper + combiner: satu split -> {kata: set(id)}."""
    d = pd.read_parquet(path, columns=["id", "judul"])
    kategori = path.parent.name.split("=")[1]
    lokal = collections.defaultdict(set)
    for nomor, (i, judul) in enumerate(zip(d["id"], d["judul"]), start=1):
        doc = id_dokumen(i, kategori, nomor)
        for kata in set(tokenisasi(judul)):      # set: satu kata dihitung sekali per judul
            lokal[kata].add(doc)
    return lokal

def reduce_index(daftar_lokal):
    """Shuffle + reduce: gabungkan id per kata dari semua mapper."""
    gabungan = collections.defaultdict(set)
    for lokal in daftar_lokal:
        for kata, ids in lokal.items():
            gabungan[kata] |= ids
    return {k: sorted(v, key=kunci_urut) for k, v in gabungan.items()}

indeks = reduce_index([mapper_index(p) for p in splits])
print("Kata dalam indeks:", f"{len(indeks):,}", "(harus sama dengan kata unik H-9:", f"{len(hasil_semua):,})")

for kata in ["corona", "virus", "jokowi"]:
    ids = indeks[kata]
    print(f"{kata:<8} muncul di {len(ids):>6,} judul | 5 id pertama: {ids[:5]}")

# Cek silang: jumlah id per kata = jumlah judul yang memuat kata itu (cara biasa)
cek_df = collections.Counter()
for t in bersih["judul"]:
    for kata in set(tokenisasi(t)):
        cek_df[kata] += 1
print("\nJumlah id per kata sama dengan cek biasa:", all(len(v) == cek_df[k] for k, v in indeks.items()))

# Contoh kata dari kategori inet (id pengganti)
inet_ids = [i for i in indeks["ponsel"] if isinstance(i, str)][:3]
print("Contoh id pengganti kategori inet (kata 'ponsel'):", inet_ids)

# Simpan ke lapisan siap_pakai
with open(SIAP / "inverted_index.jsonl", "w", encoding="utf-8") as f:
    for kata in sorted(indeks):
        f.write(json.dumps({"kata": kata, "jumlah_judul": len(indeks[kata]), "id": indeks[kata]},
                           ensure_ascii=False) + "\n")
print("Tersimpan:", SIAP / "inverted_index.jsonl")

Kata dalam indeks: 32,275 (harus sama dengan kata unik H-9: 32,275)
corona   muncul di 17,228 judul | 5 id pertama: [4856495, 4856598, 4868317, 4868545, 4869163]
virus    muncul di  3,067 judul | 5 id pertama: [4856495, 4856598, 4860201, 4865102, 4865341]
jokowi   muncul di  2,316 judul | 5 id pertama: [4842674, 4842690, 4842703, 4842774, 4843215]

Jumlah id per kata sama dengan cek biasa: True
Contoh id pengganti kategori inet (kata 'ponsel'): ['inet-000046', 'inet-000076', 'inet-000081']
Tersimpan: siap_pakai/inverted_index.jsonl


# Simpan Artefak (output) ke GDrive

In [24]:
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount("/content/drive")

TUJUAN = Path("/content/drive/MyDrive/2411533007_DerielChaerahman_BigData/Praktikum5")
TUJUAN.mkdir(parents=True, exist_ok=True)

# Folder artefak
for folder in ["lapisan_mentah", "lapisan_terkurasi", "siap_pakai", "contoh_crawl"]:
    if os.path.exists(folder):
        shutil.copytree(folder, TUJUAN / folder, dirs_exist_ok=True)

# File artefak
for berkas in ["masukan.tsv", "mapper.py", "reducer.py", "hasil_pipe.txt",
               "BD_P05_2411533007_lapisan.zip"]:
    if os.path.exists(berkas):
        shutil.copy2(berkas, TUJUAN / berkas)

print("Tersimpan di:", TUJUAN)
for p in sorted(TUJUAN.iterdir()):
    print(" -", p.name)

MessageError: Error: credential propagation was unsuccessful